In [3]:
# Retrieve kinematic data for an episode

from datasets import load_dataset

repo_id = "lerobot/full_folding"
hf_dataset = load_dataset(repo_id, streaming=True, split="train")

hf_episode_one_samples = []

for sample in iter(hf_dataset):
  if sample['episode_index'] == 0:
    hf_episode_one_samples.append(sample)
  else:
    break

In [5]:
len(hf_episode_one_samples)

1505

In [6]:
import time
import numpy as np
import mujoco
import mujoco.viewer
from datasets import load_dataset
from kinematic_mappers import map_full_folding_dataset_sample_to_openarm_v1_mujoco_qpos

repo_id = "lerobot/full_folding"
mjcf_path = "../../arms/openarm_mujoco/v1/scene.xml"

# Load model and dataset
model = mujoco.MjModel.from_xml_path(mjcf_path)
data = mujoco.MjData(model)

# Create a clean lookup table matching MuJoCo joint names to their qpos memory address
mujoco_joint_adrs = {}
for i in range(model.njnt):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i)
    mujoco_joint_adrs[name] = model.jnt_qposadr[i]

# Launch the corrected loop
with mujoco.viewer.launch_passive(model, data) as viewer:
    time_step = 1.0 / 30.0
    frame_index = 0

    # Initial camera position
    viewer.cam.lookat = [0.07, 0.03, 0.47]
    viewer.cam.azimuth = -0.2
    viewer.cam.elevation = -87.9
    viewer.cam.distance = 0.88

    for sample in hf_episode_one_samples:
        if not viewer.is_running():
            break
        loop_start = time.time()

        # actual_state = sample["observation.state"]

        # Reset velocities to stop any accumulated physics drifting
        data.qvel[:] = 0.0

        # Route the data explicitly through your index map
        # for ds_idx, qpos_adr in idx_map.items():
        #     data.qpos[qpos_adr] = math.radians(actual_state[ds_idx])
        data.qpos = map_full_folding_dataset_sample_to_openarm_v1_mujoco_qpos(sample)

        # Retrieve current camera position
        lookat = viewer.cam.lookat
        azimuth = viewer.cam.azimuth
        elevation = viewer.cam.elevation
        distance = viewer.cam.distance

        text_overlay = [
            (mujoco.mjtGridPos.mjGRID_TOPLEFT, 0, f"Current Frame: {frame_index}", ""),
            (mujoco.mjtGridPos.mjGRID_TOPLEFT, 1, f"Sim Time: {(frame_index * time_step):0.2f}s", ""),
            # (mujoco.mjtGridPos.mjGRID_TOPLEFT, 0, f"LookAt: [{lookat[0]:.2f}, {lookat[1]:.2f}, {lookat[2]:.2f}]", ""),
            # (mujoco.mjtGridPos.mjGRID_TOPLEFT, 1, f"Az/El: {azimuth:.1f} / {elevation:.1f}", ""),
            # (mujoco.mjtGridPos.mjGRID_TOPLEFT, 2, f"Dist: {distance:.2f}m", "")
        ]
        viewer.set_texts(text_overlay)

        # Run forward position kinematics
        mujoco.mj_fwdPosition(model, data)
        viewer.sync()

        frame_index += 1
        elapsed = time.time() - loop_start
        if elapsed < time_step:
            time.sleep(time_step - elapsed)

actual_state [-9.14712905883789, -0.9507768154144287, 0.010928469710052013, 9.780980110168457, -0.44806724786758423, -0.07649928331375122, -0.05464234575629234, -0.33878254890441895, 9.54055404663086, -0.3606394827365875, -0.6010658144950867, 11.245394706726074, -0.0983562245965004, 0.7759213447570801, -0.22949786484241486, -26.588966369628906]
actual_state [-9.168986320495605, -0.9507768154144287, 0.010928469710052013, 9.780980110168457, -0.44806724786758423, -0.07649928331375122, -0.032785408198833466, -0.33878254890441895, 9.54055404663086, -0.33878254890441895, -0.6010658144950867, 11.245394706726074, 0.07649928331375122, 0.7759213447570801, -0.22949786484241486, -37.25515365600586]
actual_state [-9.168986320495605, -0.9507768154144287, 0.010928469710052013, 9.780980110168457, -0.44806724786758423, -0.07649928331375122, -0.010928469710052013, -0.33878254890441895, 9.54055404663086, -0.3606394827365875, -0.31692561507225037, 11.245394706726074, -0.0983562245965004, 0.775921344757080